In [2]:
import forgethreads as ft
import numpy as np
from numba import njit


In [ ]:

rng = np.random.default_rng()
ft.set_verbose(False)


In [ ]:
# n= 100
# distribution = 0.7

# m_size = n ** 2 



# # Llenar segun distribucion

# m1= np.zeros(shape=(1,n,n))

# total_for_fill =  (n * (n-1)) / 2
# total_for_fill_pondered = total_for_fill * distribution

# ni = 0
# while(ni < total_for_fill_pondered):
#   o = rng.integers(0,n)
#   d = rng.integers(0,n)
#   if(m1[0,o,d]) == 0:
#     m1[0,o,d] = 1
#     m1[0,d,o] = 1
#     ni += 1



# m2 = m1.copy()
# paths,values, max_order = ft.maxmin(m1,m2,0.1,200,False)

In [ ]:


@njit(cache=True, fastmath=True)
def _flat_to_rc(sel, n, m, rows, cols):
    """
    Mapea índices planos [0, total_possible) → (row, col) en M sin
    construir la tabla completa de índices.

    Partición:
      [0,        n*(n-1))           → bloque A (n×n) fuera de diagonal
      [n*(n-1),  n*(n-1) + n*m)     → bloque B (n×m)
      [n*(n-1) + n*m, total)        → bloque D (m×m) fuera de diagonal
    """
    nA = n * (n - 1)
    nB = n * m
    n1 = n - 1 if n > 1 else 1   # guarda contra /0 cuando n==1 (nA=0)
    m1 = m - 1 if m > 1 else 1

    for i in range(len(sel)):
        f = sel[i]
        if f < nA:                         # ── bloque A
            r = f // n1
            c = f % n1
            if c >= r:
                c += 1
            rows[i] = r
            cols[i] = c
        elif f < nA + nB:                  # ── bloque B
            f -= nA
            rows[i] = f // m
            cols[i] = n + f % m
        else:                              # ── bloque D
            f -= nA + nB
            r  = f // m1
            c  = f % m1
            if c >= r:
                c += 1
            rows[i] = n + r
            cols[i] = n + c


def build_matrix2(n, m, d, seed=None):
    rng = np.random.default_rng(seed)
    N   = n + m
    h   = np.float32(0.5)

    # ── 1. Bloques directamente en float32 (evita float64 intermedio) ─
    M = np.zeros((N, N), dtype=np.float32)
    M[:n, :n] = rng.random((n, n), dtype=np.float32) * h   # [0, 0.5)
    M[n:, n:] = rng.random((m, m), dtype=np.float32) * h
    M[:n, n:] = rng.random((n, m), dtype=np.float32) * h



    # ── 2. Diagonales = 1 sobre la vista (sin copias) ─────────────────
    np.fill_diagonal(M[:n, :n], 1.0)
    np.fill_diagonal(M[n:, n:], 1.0)

    # ── 3. Aristas extra ───────────────────────────────────────────────
    # current_edges es siempre 0 → todos los valores generados son < 0.5
    total_possible = n * (n - 1) + n * m + m * (m - 1)
    target_edges   = min(int(round(d * n)), total_possible)

    if target_edges > 0:
        # choice usa Floyd's sampling → O(k) cuando k << total
        sel  = rng.choice(total_possible, size=target_edges, replace=False)
        rows = np.empty(target_edges, dtype=np.int64)
        cols = np.empty(target_edges, dtype=np.int64)
        _flat_to_rc(sel, n, m, rows, cols)                  # JIT compilado
        M[rows, cols] = rng.random(target_edges, dtype=np.float32) * h + h
    else: 
      print("The distribution is incompatible ")
    return M.reshape(1, N, N).astype(np.float16)
    # return M

# ── Warm-up: compilar el JIT antes del bucle principal ────────────────
build_matrix2(4, 2, 1.0, seed=0)

    
    

In [ ]:
m1 = build_matrix2(100,100,np.log(1000),1111)

In [ ]:
def run_experimental(n, avg_degree,i):
    m1 = build_matrix2(n,n,avg_degree,1111)
    m2 = m1.copy()
    paths, values, max_order = ft.maxmin(
      m1.reshape(1, n *2, n*2).astype(np.float16),
      m2.reshape(1, n * 2, n*2).astype(np.float16),
      0.5, 6,False)
    # if i ==0:
    #   print(paths.to_numpy())
    return max_order

In [ ]:
np.log10(10000)

In [ ]:
ns = [1000]
iterations = 10
avg_degrees = []
proms_by_n = []
for n in ns:
    m = n    
    avg_degrees = np.arange(np.log10(n*2),np.log10(n*2)  +1,step=0.1) 
    print(avg_degrees)
    proms = []
    print(f"calculating  : n={n} ...", end="\t")

    for deg in avg_degrees:
        # print(f"calculating deg={deg} : n={n} ...", end="\t")
        acum = 0
        for i in range(iterations):
            acum += run_experimental(n, deg,i)
        proms.append(acum / iterations)
    print(f"\ncalculated averages for {n} nodes")
    proms_by_n.append(proms)

# p300 = np.load("proms_300n.npy")
# p400 = np.load("proms_400n.npy")
# p500 = np.load("proms_500n.npy")

# proms_by_n.append(p300[0]) E(eta_0) < b log(N), 
# proms_by_n.append(p400[0])
# proms_by_n.append(p500[0])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def plot_proms(ns, avg_degrees, proms_by_n):
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for i, n in enumerate(ns):
        color = colors[i % len(colors)]
        ax.plot(avg_degrees, proms_by_n[i], 'o-', label=f"N = {n}",
                linewidth=2, markersize=5, color=color)

    ad = np.trunc(avg_degrees * 1000) / 1000
    # ax.set_xticklabels([str( d) for d in ad], rotation=30, fontsize=9)
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())
    ax.set_xlabel("Grado promedio (avg_degree)", fontsize=12)
    ax.set_ylabel("E[η₀] — orden máximo promedio", fontsize=12)
    ax.set_title("Orden máximo promedio vs densidad del grafo", fontsize=13)

    # Leyenda combinada: curva + ln(n)
    handles, labels = ax.get_legend_handles_labels()
    for i, n in enumerate(ns):
        color = colors[i % len(colors)]
        handles.append(plt.Line2D([0], [0], color=color, linestyle=':', linewidth=1.5, alpha=0.7))

    ax.legend(handles=handles, labels=labels, title="Nodos N", fontsize=10, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_proms(ns, avg_degrees, proms_by_n)